# ChuckleNet: Train on Existing Data (191K samples)

**Data already on Drive:**
- `vtt_splits/train.npz`: 191,495 samples (WavLM 768-dim)
- `vtt_splits/valid.npz`: 26,807 samples
- `vtt_splits/test.npz`: 17,950 samples
- `wavlm_embeddings/`: Per-video embeddings (2147 videos)

**Goal**: Train fusion model on existing WavLM + add prosody

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive')

!pip install -q torch numpy pandas scikit-learn

import numpy as np
import io
import subprocess

BASE = '/content/drive/MyDrive/chuckle_net'

# Load existing splits
def load_split(name):
    data = np.load(io.BytesIO(
        subprocess.run(['rclone', 'cat', f'{BASE}/vtt_splits/{name}.npz'],
                       capture_output=True).stdout
    ))
    return data['X'], data['y']

print('Loading data...')
X_train, y_train = load_split('train')
X_valid, y_valid = load_split('valid')
X_test, y_test = load_split('test')

print(f'Train: {X_train.shape}, pos={y_train.sum()}/{len(y_train)} ({100*y_train.mean():.1f}%)')
print(f'Valid: {X_valid.shape}, pos={y_valid.sum()}/{len(y_valid)} ({100*y_valid.mean():.1f}%)')
print(f'Test: {X_test.shape}, pos={y_test.sum()}/{len(y_test)} ({100*y_test.mean():.1f}%)')

In [ ]:
# === TRAIN FUSION MODEL ===
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

class WavLMClassifier(nn.Module):
    def __init__(self, dim=768, hidden=[512, 256, 64]):
        super().__init__()
        self.bn0 = nn.BatchNorm1d(dim)
        layers = []
        prev = dim
        for h in hidden:
            layers.extend([
                nn.Linear(prev, h), nn.BatchNorm1d(h),
                nn.ReLU(), nn.Dropout(0.3)
            ])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(self.bn0(x)).squeeze(-1)

# Convert to tensors
X_tr = torch.FloatTensor(X_train)
y_tr = torch.FloatTensor(y_train)
X_va = torch.FloatTensor(X_valid)
y_va = torch.FloatTensor(y_valid)
X_te = torch.FloatTensor(X_test)
y_te = torch.FloatTensor(y_test)

print(f'\nTraining on {len(X_tr)} samples...')

# Training
model = WavLMClassifier(dim=768, hidden=[512, 256, 64]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
pos_w = torch.tensor((1-y_tr.mean())/max(0.01, y_tr.mean())).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)

best_f1, best_state = 0, None
batch_size = 512

for epoch in range(50):
    # Train
    model.train()
    for i in range(0, len(X_tr), batch_size):
        batch_x = X_tr[i:i+batch_size].to(device)
        batch_y = y_tr[i:i+batch_size].to(device)
        opt.zero_grad()
        out = model(batch_x)
        loss = loss_fn(out, batch_y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    sched.step()
    
    # Eval
    model.eval()
    with torch.no_grad():
        pred_va = (torch.sigmoid(model(X_va.to(device))) > 0.5).cpu().numpy().astype(int)
        f1_va = f1_score(y_va.numpy(), pred_va, zero_division=0)
        
        pred_te = (torch.sigmoid(model(X_te.to(device))) > 0.5).cpu().numpy().astype(int)
        f1_te = f1_score(y_te.numpy(), pred_te, zero_division=0)
        prec = precision_score(y_te.numpy(), pred_te, zero_division=0)
        rec = recall_score(y_te.numpy(), pred_te, zero_division=0)
    
    if f1_va > best_f1:
        best_f1 = f1_va
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: Val F1={f1_va:.4f}, Test F1={f1_te:.4f} (P={prec:.3f}, R={rec:.3f})')

print(f'\nBest Val F1: {best_f1:.4f}')

# Final evaluation
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    pred_te = (torch.sigmoid(model(X_te.to(device))) > 0.5).cpu().numpy().astype(int)
    f1 = f1_score(y_te.numpy(), pred_te, zero_division=0)
    prec = precision_score(y_te.numpy(), pred_te, zero_division=0)
    rec = recall_score(y_te.numpy(), pred_te, zero_division=0)

print(f'\n=== TEST RESULTS ===')
print(f'F1: {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Reacall: {rec:.4f}')

# Save
torch.save(best_state, f'{BASE}/wavlm_model.pt')
print(f'\nModel saved: {BASE}/wavlm_model.pt')